In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [2]:
model_name = "Qwen/Qwen3-8B"

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to("cuda:3")

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [11]:
text = "I love Russia"
messages = [
                {"role": "user", "content": text},
            ]
text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False, enable_thinking=False)

In [12]:
input_ids = tokenizer(text, return_tensors='pt').to("cuda:3")

In [13]:
input_ids

{'input_ids': tensor([[151644,    872,    198,     40,   2948,   8359, 151645,    198, 151644,
          77091,    198, 151667,    271, 151668,    271]], device='cuda:3'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:3')}

In [14]:
generation = model.generate(**input_ids)

In [15]:
generation

tensor([[151644,    872,    198,     40,   2948,   8359, 151645,    198, 151644,
          77091,    198, 151667,    271, 151668,    271,   4792,    594,  11117,
            311,   6723,      0,   8359,    374,    264,  26291,   3146,    448,
            264,   9080,   3840,     11,  16807,   7674,     11,    323]],
       device='cuda:3')

In [16]:
tokenizer.decode(input_ids["input_ids"][0])

'<|im_start|>user\nI love Russia<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'

In [17]:
tokenizer.decode(generation[0])

"<|im_start|>user\nI love Russia<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nThat's wonderful to hear! Russia is a fascinating country with a rich history, diverse culture, and"

In [4]:
type(tokenizer)

transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast

In [27]:
type(input_ids)

transformers.tokenization_utils_base.BatchEncoding

In [24]:
input_ids.input_ids.shape[1]

3

In [16]:
type(input_ids["input_ids"])

torch.Tensor

In [10]:
input_ids["input_ids"].shape[1]

3

In [26]:
tokenizer.decode(input_ids["input_ids"][0])

'<s> I love Russia'

In [11]:
messages = [
    {"role": "user", "content": "I love Russia"}
]

In [12]:
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False # Switches between thinking and non-thinking modes. Default is True.
)

In [13]:
text

'<|im_start|>user\nI love Russia<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'

In [5]:
input_ids

tensor([[  40, 2948, 8359]])

In [6]:
tokenizer.decode(input_ids[0])

'I love Russia'

In [77]:
import json

In [91]:
with open('/app/data/raw/SQuAD/preprocessed.json') as f:
    data = json.load(f)

In [94]:
no_answer = 0
for sample in data:
    if len(sample['answers']) == 0:
        no_answer += 1

In [95]:
no_answer

0

In [100]:
from datasets import load_dataset
import random

In [98]:
dataset = load_dataset("EdinburghNLP/xsum")

default/train/0000.parquet:   0%|          | 0.00/304M [00:00<?, ?B/s]

default/validation/0000.parquet:   0%|          | 0.00/16.7M [00:00<?, ?B/s]

default/test/0000.parquet:   0%|          | 0.00/17.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/204045 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11332 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11334 [00:00<?, ? examples/s]

In [102]:
data = random.sample(list(dataset["validation"]), k=1500)

In [4]:
import json

with open("/app/data/fixed/SQuAD/preprocessed.json", "r") as f:
    d = json.load(f)

In [5]:
len(d)

5928

In [104]:
ids = set([elem["id"] for elem in data])
len(ids) == len(data)

True

In [4]:
import pandas as pd
import ast

In [51]:
df_ = pd.read_csv("/app/data/fixed/CoQA/Mistral-7B-Instruct-v0.1/generations.csv")

In [10]:
ast.literal_eval(df_.iloc[0]["answers"])[0]

'white'

In [11]:
df_.iloc[0]

context             Once upon a time, in a barn near a farm house,...
question                                       What color was Cotton?
answers                          ['white', 'white', 'white', 'white']
id                                   3dr23u6we5exclen4th8uq9rb42tel_0
prompt              <s> Once upon a time, in a barn near a farm ho...
generated_answer                                              White. 
Name: 0, dtype: object

In [52]:
df = pd.read_csv("/app/data/fixed/SQuAD/Mistral-7B-Instruct-v0.1/generations_with_rougel.csv")

In [53]:
import random

for i in random.sample(range(len(df)), 5):
    print(df.iloc[i])
    print(df.iloc[i]["context"])
    print(df.iloc[i]["question"])
    print(df.iloc[i]["answers"])
    print(df.iloc[i]["generated_answer"])
    print(df.iloc[i]["rougel"])

id                                           5726a14c708984140094cc52
title                                              European_Union_law
context             The concept of legal certainty is recognised o...
question                         Which laws mentioned predate EU law?
answers                          ['international law and public law']
prompt              <s> [INST] Given the context, answer the quest...
generated_answer    The concepts of legal certainty and good faith...
rougel                                                        0.15625
Name: 1627, dtype: object
The concept of legal certainty is recognised one of the general principles of European Union law by the European Court of Justice since the 1960s. It is an important general principle of international law and public law, which predates European Union law. As a general principle in European Union law it means that the law must be certain, in that it is clear and precise, and its legal implications foreseeable, s

In [39]:
df.iloc[400]["generated_answer"]

'There are two extended metropolitan areas.'

In [36]:
df.iloc[400]

id                                           5706074552bb8914006897d5
title                                             Southern_California
context             Southern California consists of one Combined S...
question              How many extended metropolitan areas are there?
answers                                                       ['two']
prompt              <|im_start|>user\nGiven the context, answer th...
generated_answer           There are two extended metropolitan areas.
rougel                                                            1.0
Name: 400, dtype: object

In [28]:
len(df)

5928

In [1]:
import pandas as pd

In [5]:
model_names = ["Llama-2-7b-chat-hf", "Llama-2-13b-chat-hf", "Llama-3.1-8B-Instruct", "Qwen3-8B", "Mistral-7B-Instruct-v0.1"]

In [7]:
df.head()

,Unnamed: 0,id,title,context,question,answers,prompt,generated_answer,rougel,gpt_annotation,gpt_label
0,4445,572fec30947a6a140053cdf5,Rhine,"In the centre of Basel, the first major city i...",What ends at this bend in the Rhine?,['High Rhine'],"<s> [INST] Given the context, answer the quest...",The High Rhine ends at the bend in the Rhine.,1.000000,No,0.0
1,2372,5726577f708984140094c303,Black_Death,The Black Death ravaged much of the Islamic wo...,How long did plague last in the Ottoman empire?,['until the second quarter of the 19th century...,"<s> [INST] Given the context, answer the quest...",The plague repeatedly struck the Ottoman empir...,0.113636,Yes,1.0
2,2642,5728d63c4b864d1900164f1b,Civil_disobedience,"Following the Peterloo massacre of 1819, poet ...",Which famous Indian practiced civil disobedience?,['Gandhi'],"<s> [INST] Given the context, answer the quest...",Mahatma Gandhi practiced civil disobedience.,1.000000,No,0.0
3,165,56e1b62ecd28a01900c67aa6,Computational_complexity_theory,For a precise definition of what it means to s...,What is the most critical resource measured to...,['time'],"<s> [INST] Given the context, answer the quest...",The most critical resource measured to assess ...,0.500000,Yes,1.0
4,2240,5726431aec44d21400f3dd17,Packet_switching,Tymnet was an international data communication...,Gateways allowed private companies to do what,"['reach locations not on the private network',...","<s> [INST] Given the context, answer the quest...",Gateways allowed private companies to build th...,0.619048,No,0.0


In [6]:
for model_name in model_names:    
    df = pd.read_csv(f"/app/data/fixed/SQuAD/squad_{model_name}.csv")
    print(df.iloc[0]["prompt"])

<s> [INST] Given the context, answer the question in a single brief but complete sentence. Note that your answer should be strictly based on the given context. In case the context does not contain the necessary information to answer the question, please reply with: "Unable to answer based on given context."
Context: Fresno has three large public parks, two in the city limits and one in county land to the southwest. Woodward Park, which features the Shinzen Japanese Gardens, numerous picnic areas and several miles of trails, is in North Fresno and is adjacent to the San Joaquin River Parkway. Roeding Park, near Downtown Fresno, is home to the Fresno Chaffee Zoo, and Rotary Storyland and Playland. Kearney Park is the largest of the Fresno region's park system and is home to historic Kearney Mansion and plays host to the annual Civil War Revisited, the largest reenactment of the Civil War in the west coast of the U.S.
Question: Which park hosts the largest Civil War reenactment on the wes